In [1]:
import sys
print(sys.executable)

/Users/sarah/Desktop/Final Project/my-project/.venv/bin/python


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
print(pd.__version__)

3.0.6


In [3]:
import ast
import pandas as pd

In [4]:
import os
print(os.getcwd())
print(os.listdir(".."))

/Users/sarah/Desktop/Final Project/my-project/notebooks
['.DS_Store', 'requirements.txt', 'README.md', '.gitignore', '.venv', '.ipynb_checkpoints', '.git', 'data', 'notebooks', 'src']


In [5]:
df_original = pd.read_csv("../data/raw/recipes.csv")

In [6]:
df = df_original.copy()

In [7]:
df[["RecipeIngredientQuantities", "RecipeIngredientParts"]].head(10)

,RecipeIngredientQuantities,RecipeIngredientParts
0,"c(""4"", ""1/4"", ""1"", ""1"")","c(""blueberries"", ""granulated sugar"", ""vanilla ..."
1,"c(""1"", ""4"", ""2"", ""2"", ""8"", ""1/4"", ""8"", ""1/2"", ...","c(""saffron"", ""milk"", ""hot green chili peppers""..."
2,"c(""1 1/2"", ""1"", NA, ""1 1/2"", NA, ""3/4"")","c(""sugar"", ""lemons, rind of"", ""lemon, zest of""..."
3,"c(""12"", ""1"", ""2"", ""1"", ""10"", ""1"", ""3"", ""2"", ""2...","c(""extra firm tofu"", ""eggplant"", ""zucchini"", ""..."
4,"c(""46"", ""4"", ""1"", ""2"", ""1"")","c(""plain tomato juice"", ""cabbage"", ""onion"", ""c..."
5,"c(""1 1/4"", ""1/4"", ""6"", ""1/3"", ""1/4"", ""1/4"", ""2...","c(""graham cracker crumbs"", ""sugar"", ""butter"", ..."
6,"c(""12"", ""2"", ""3"", ""450"", ""1"", ""2"", ""1/4"", ""1"",...","c(""chicken"", ""butter"", ""flour"", ""milk"", ""celer..."
7,"c(""3/4"", ""1"", ""1"", ""2"", ""3"", ""1/4"", ""1"", ""1/2""...","c(""sugar"", ""margarine"", ""egg"", ""flour"", ""salt""..."
8,"c(""1/2"", ""5"", ""2"", ""1"", ""1"", ""1"")","c(""rice vinegar"", ""haeo"")"
9,"c(""3/4"", ""1/2"", ""1"", ""1"", ""1"", ""2"", ""1"")","c(""butter"", ""brown sugar"", ""granulated sugar"",..."


In [8]:
import os
import re
from collections import Counter
from fractions import Fraction

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

In [9]:

recipes = df
print(recipes.shape)
recipes.head(3)

(522517, 28)


,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,...,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeYield,RecipeInstructions
0,38,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09T21:46:00Z,Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.,"c(""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive...",...,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0,NaN,"c(""Toss 2 cups berries with sugar."", ""Let stand for 45 minutes, stirring occ..."
1,39,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29T13:12:00Z,Make and share this Biryani recipe from Food.com.,"c(""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive...",...,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0,NaN,"c(""Soak saffron in warm milk for 5 minutes and puree in blender."", ""Add chil..."
2,40,Best Lemonade,1566,Stephen Little,PT5M,PT30M,PT35M,1999-09-05T19:52:00Z,This is from one of my first Good House Keeping cookbooks. You must use a ...,"c(""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive...",...,0.0,0.0,1.8,81.5,0.4,77.2,0.3,4.0,NaN,"c(""Into a 1 quart Jar with tight fitting lid, put sugar and lemon peel, or z..."


In [10]:
# Counter for the number of recipes after each step
funnel = []

def log(step, d):
    funnel.append((step, len(d)))
    print(f"{step}: {len(d):,} rows")

log("0. raw", recipes)

0. raw: 522,517 rows


In [11]:
DROP_COLS = ["DatePublished", "Images", "Keywords", "AggregatedRating",
             "ReviewCount", "RecipeYield", "CookTime", "PrepTime"]

df = recipes.drop(columns=DROP_COLS, errors="ignore").copy()

ESSENTIAL = ["Name", "RecipeIngredientParts", "RecipeIngredientQuantities",
             "TotalTime", "Calories"]

print("Missing % (essential + servings):")
print((df[ESSENTIAL + ["RecipeServings"]].isna().mean() * 100).round(2))

df = df.dropna(subset=ESSENTIAL).copy()
df["has_servings"] = df["RecipeServings"].notna()
log("1. dropna on essential columns", df)

Missing % (essential + servings):
Name                           0.00
RecipeIngredientParts          0.00
RecipeIngredientQuantities     0.00
TotalTime                      0.00
Calories                       0.00
RecipeServings                35.01
dtype: float64
1. dropna on essential columns: 522,514 rows


In [12]:
# The new parser: reads everything between quotation marks as a single item, and NA as a missing value
_PATTERN = re.compile(r'"([^"]*)"|\b(NA)\b')

def parse_r_vector(text):
    """c("a", "b, c", NA) -> ['a', 'b, c', None]"""
    if pd.isna(text):
        return []
    out = []
    for quoted, na in _PATTERN.findall(str(text)):
        out.append(None if na else quoted)
    return out


# الـ parser القديم (للمقارنة فقط)
def parse_naive(text):
    if pd.isna(text):
        return []
    text = str(text).strip()
    if text.startswith("c(") and text.endswith(")"):
        text = text[2:-1]
    return [i.strip().strip('"').strip("'") for i in text.split(",") if i.strip()]


# Quick test on examples from the data
print(parse_r_vector('c("1 1/2", "1", NA, "1 1/2", NA, "3/4")'))
print(parse_r_vector('c("sugar", "lemons, rind of", "lemon, zest of")'))
print(parse_naive('c("sugar", "lemons, rind of", "lemon, zest of")'))

['1 1/2', '1', None, '1 1/2', None, '3/4']
['sugar', 'lemons, rind of', 'lemon, zest of']
['sugar', 'lemons', 'rind of', 'lemon', 'zest of']


In [13]:
df["parts"] = df["RecipeIngredientParts"].apply(parse_r_vector)
df["quants"] = df["RecipeIngredientQuantities"].apply(parse_r_vector)

# Compare the mismatch rate between the old and new parser
old_mismatch = (df["RecipeIngredientParts"].apply(parse_naive).str.len()
                != df["RecipeIngredientQuantities"].apply(parse_naive).str.len()).mean() * 100
new_mismatch = (df["parts"].str.len() != df["quants"].str.len()).mean() * 100

print(f"Mismatch rate, old parser: {old_mismatch:.2f}%")
print(f"Mismatch rate, new parser: {new_mismatch:.2f}%")

Mismatch rate, old parser: 77.86%
Mismatch rate, new parser: 78.00%


In [14]:
df["n_parts"] = df["parts"].str.len()
df["n_quants"] = df["quants"].str.len()
df["mismatch"] = df["n_parts"] != df["n_quants"]
df["parts_na"] = df["parts"].apply(lambda l: sum(x is None for x in l))
df["quants_na"] = df["quants"].apply(lambda l: sum(x is None for x in l))
df["empty"] = df["n_parts"] == 0

total = len(df)
print(f"Empty ingredient lists:   {df['empty'].sum():,}")
print(f"Length mismatch:          {df['mismatch'].sum():,} ({df['mismatch'].mean()*100:.2f}%)")
print(f"Recipes with NA in parts: {(df['parts_na'] > 0).sum():,}")
print(f"Recipes with NA in quants:{(df['quants_na'] > 0).sum():,} ({(df['quants_na'] > 0).mean()*100:.2f}%)")

base_ok = (~df["empty"]) & (~df["mismatch"]) & (df["parts_na"] == 0)
print()
print(f"Valid, lenient (NA quantities allowed): {base_ok.sum():,}")
print(f"Valid, strict  (no NA quantities):      {(base_ok & (df['quants_na'] == 0)).sum():,}")

Empty ingredient lists:   2,218
Length mismatch:          407,536 (78.00%)
Recipes with NA in parts: 0
Recipes with NA in quants:180,758 (34.59%)

Valid, lenient (NA quantities allowed): 114,958
Valid, strict  (no NA quantities):      86,077


In [15]:
STRICT_QUANTITIES = True 

mask = (~df["empty"]) & (~df["mismatch"]) & (df["parts_na"] == 0)
if STRICT_QUANTITIES:
    mask &= (df["quants_na"] == 0)

df = df[mask].copy()
log(f"2. valid ingredient/quantity lists (strict={STRICT_QUANTITIES})", df)

2. valid ingredient/quantity lists (strict=True): 86,077 rows


In [16]:
dup_key = (df["Name"].str.lower().str.strip() + "|"
           + df["parts"].apply(lambda l: ",".join(sorted(x.lower() for x in l))))

print(f"Duplicates (name + ingredients): {dup_key.duplicated().sum():,}")
df = df[~dup_key.duplicated()].copy()
log("3. drop duplicates (name + ingredients)", df)

Duplicates (name + ingredients): 533
3. drop duplicates (name + ingredients): 85,544 rows


In [17]:
def parse_duration(s):
    """'PT1H10M' -> 70 ; 'P1DT2H' -> 1560"""
    if pd.isna(s):
        return np.nan
    s = str(s)
    d = re.search(r"(\d+)D", s)
    h = re.search(r"(\d+)H", s)
    m = re.search(r"(\d+)M", s)
    return ((int(d.group(1)) if d else 0) * 1440
            + (int(h.group(1)) if h else 0) * 60
            + (int(m.group(1)) if m else 0))

df["TotalTime_min"] = df["TotalTime"].apply(parse_duration)
df = df.drop(columns=["TotalTime"])

print(df["TotalTime_min"].describe(percentiles=[.5, .95, .99, .999]).round(1))
print(f"\nRecipes over 1440 min (1 day): {(df['TotalTime_min'] > 1440).sum():,}")

df = df[df["TotalTime_min"] > 0].copy()
log("4. drop TotalTime == 0", df)

count       85544.0
mean          708.4
std        149044.2
min             0.0
50%            35.0
95%           250.0
99%          1450.0
99.9%       30252.3
max      43552800.0
Name: TotalTime_min, dtype: float64

Recipes over 1440 min (1 day): 957
4. drop TotalTime == 0: 85,152 rows


In [18]:
NUTRITION_COLS = ["Calories", "FatContent", "SaturatedFatContent", "CholesterolContent",
                  "SodiumContent", "CarbohydrateContent", "FiberContent",
                  "SugarContent", "ProteinContent"]

print("Zero % per nutrition column:")
print(((df[NUTRITION_COLS] == 0).mean() * 100).round(2))

df = df[df["Calories"] > 0].copy()
log("5. drop Calories == 0", df)

df[NUTRITION_COLS].describe(percentiles=[.5, .95, .99, .999]).T.round(1)

Zero % per nutrition column:
Calories                0.29
FatContent              1.97
SaturatedFatContent     5.93
CholesterolContent     23.89
SodiumContent           0.25
CarbohydrateContent     0.55
FiberContent            5.39
SugarContent            2.09
ProteinContent          1.34
dtype: float64
5. drop Calories == 0: 84,905 rows


,count,mean,std,min,50%,95%,99%,99.9%,max
Calories,84905.0,461.1,725.4,0.1,279.9,1439.6,3663.8,8017.0,23001.1
FatContent,84905.0,22.2,43.2,0.0,11.3,75.5,204.5,479.7,1997.2
SaturatedFatContent,84905.0,9.2,19.5,0.0,4.0,30.9,93.8,238.6,777.8
CholesterolContent,84905.0,75.7,155.0,0.0,30.5,279.0,668.7,1764.9,5801.0
SodiumContent,84905.0,808.9,4998.5,0.0,314.0,2351.7,6874.6,56590.3,704129.6
CarbohydrateContent,84905.0,52.9,103.0,0.0,28.2,179.3,512.4,1076.5,5030.7
FiberContent,84905.0,3.7,6.6,0.0,1.9,12.9,27.6,64.9,472.0
SugarContent,84905.0,26.5,74.3,0.0,7.4,102.9,345.9,822.1,4570.9
ProteinContent,84905.0,13.7,22.2,0.0,6.3,47.6,84.7,201.8,1676.0


In [19]:
funnel_df = pd.DataFrame(funnel, columns=["step", "n_recipes"])
funnel_df["lost_vs_previous"] = (-funnel_df["n_recipes"].diff()).fillna(0).astype(int)
funnel_df["% of raw"] = (funnel_df["n_recipes"] / funnel_df["n_recipes"].iloc[0] * 100).round(2)
funnel_df

,step,n_recipes,lost_vs_previous,% of raw
0,0. raw,522517,0,100.00
1,1. dropna on essential columns,522514,3,100.00
2,2. valid ingredient/quantity lists (strict=True),86077,436437,16.47
3,3. drop duplicates (name + ingredients),85544,533,16.37
4,4. drop TotalTime == 0,85152,392,16.30
5,5. drop Calories == 0,84905,247,16.25


In [20]:
def parse_qty(q):
    """'1 1/2' -> 1.5 ; '1/4' -> 0.25 ; unparseable -> NaN"""
    if q is None:
        return np.nan
    parts = str(q).strip().split()
    if not parts:
        return np.nan
    try:
        return float(sum(Fraction(p) for p in parts))
    except (ValueError, ZeroDivisionError):
        return np.nan

df["quants_num"] = df["quants"].apply(lambda l: [parse_qty(x) for x in l])

has_nan = df["quants_num"].apply(lambda l: any(np.isnan(x) for x in l))
print(f"Recipes with at least one unparseable quantity: {has_nan.sum():,} ({has_nan.mean()*100:.2f}%)")

bad = Counter(q for qs in df.loc[has_nan, "quants"] for q in qs
              if q is not None and np.isnan(parse_qty(q)))
print("Most common unparseable strings:", bad.most_common(15))

Recipes with at least one unparseable quantity: 3,056 (3.60%)
Most common unparseable strings: [('1/2 - 1', 683), ('1/4 - 1/2', 644), ('1  - 1 1/2', 290), ('1/2 - 3/4', 251), ('1 1/2 - 2', 245), ('1/8 - 1/4', 169), ('3/4 - 1', 166), ('2 1/2 - 3', 108), ('1/3 - 1/2', 102), ('2  - 2 1/2', 77), ('1/4 - 1/3', 73), ('3  - 3 1/2', 52), ('1/2 - 2/3', 45), ('3 1/2 - 4', 45), ('4  - 4 1/2', 33)]


In [21]:
# Recompute the number of ingredients from the lists themselves
df["n_ingredients"] = df["parts"].str.len()

# Sample check: ingredient and quantity side by side
for _, row in df.sample(3, random_state=0).iterrows():
    print(row["Name"], "| servings:", row["RecipeServings"], "| kcal:", row["Calories"])
    print("   ", list(zip(row["parts"], row["quants"])))

Green Jello Salad Thingy | servings: 10.0 | kcal: 339.7
    [('lime Jell-O gelatin', '1'), ('Cool Whip', '1'), ('crushed pineapple', '1'), ('mini marshmallows', '1 1/2'), ('cottage cheese', '1'), ('walnuts', '1/2')]
Vegetarian Chipotle Chili | servings: nan | kcal: 528.4
    [('olive oil', '1'), ('carrots', '1 1/2'), ('red bell pepper', '1'), ('green bell pepper', '1'), ('onion', '1'), ('garlic', '2'), ('chili powder', '1'), ('ground cumin', '2'), ('tomatoes with juice', '1'), ('red kidney beans', '1'), ('cannellini beans', '1'), ('black beans', '1'), ('chipotle chile in adobo', '1 -2')]
Fettuccine With Shrimp, Mussels and Cherry Tomatoes | servings: 3.0 | kcal: 729.9
    [('olive oil', '2'), ('cherry tomatoes', '350'), ('sea salt', '1/2'), ('ground black pepper', '1/4'), ('garlic cloves', '4'), ('crushed red pepper flakes', '1/2'), ('white wine', '1/2'), ('celery', '1'), ('shrimp', '350'), ('mussels', '350'), ('fresh basil', '1/2'), ('fresh rosemary', '1/2'), ('fresh oregano', '2'), (

In [22]:
FINAL_COLS = ["RecipeId", "Name", "RecipeCategory", "TotalTime_min", "RecipeServings", "has_servings",
              "parts", "quants", "quants_num", "n_ingredients"] + NUTRITION_COLS
FINAL_COLS = [c for c in FINAL_COLS if c in df.columns]

clean = df[FINAL_COLS].reset_index(drop=True)
clean.to_pickle("recipes_clean.pkl")   

print(clean.shape)
clean.head()

(84905, 19)


,RecipeId,Name,RecipeCategory,TotalTime_min,RecipeServings,has_servings,parts,quants,quants_num,n_ingredients,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent
0,38,Low-Fat Berry Blue Frozen Dessert,Frozen Desserts,1485,4.0,True,"[blueberries, granulated sugar, vanilla yogurt, lemon juice]","[4, 1/4, 1, 1]","[4.0, 0.25, 1.0, 1.0]",4,170.9,2.5,1.3,8.0,29.8,37.1,3.6,30.2,3.2
1,42,Cabbage Soup,Vegetable,50,4.0,True,"[plain tomato juice, cabbage, onion, carrots, celery]","[46, 4, 1, 2, 1]","[46.0, 4.0, 1.0, 2.0, 1.0]",5,103.6,0.4,0.1,0.0,959.3,25.1,4.8,17.7,4.3
2,49,Chicken Breasts Lombardi,Chicken Breast,75,6.0,True,"[fresh mushrooms, butter, boneless skinless chicken breast halves, flour, bu...","[2, 2, 12, 1/2, 1/3, 3/4, 1/2, 1/2, 1/2, 1/2, 1/4]","[2.0, 2.0, 12.0, 0.5, 0.3333333333333333, 0.75, 0.5, 0.5, 0.5, 0.5, 0.25]",11,627.7,24.9,13.0,203.0,848.9,13.7,0.6,2.0,57.9
3,55,Betty Crocker's Southwestern Guacamole Dip,Southwestern U.S.,125,4.0,True,"[avocados, tomatoes, salt, garlic, lime juice]","[5, 1, 1/2, 4, 1/4]","[5.0, 1.0, 0.5, 4.0, 0.25]",5,415.9,36.9,5.4,0.0,310.6,24.9,17.3,2.8,5.5
4,56,Buttermilk Pie,Pie,80,8.0,True,"[butter, margarine, sugar, flour, eggs, salt, vanilla, buttermilk]","[1/2, 1 1/2, 3, 3, 1, 1, 1, 1]","[0.5, 1.5, 3.0, 3.0, 1.0, 1.0, 1.0, 1.0]",8,395.9,19.1,9.8,101.5,269.8,52.2,0.7,39.9,5.0


In [23]:
print(recipes.columns.tolist())

['RecipeId', 'Name', 'AuthorId', 'AuthorName', 'CookTime', 'PrepTime', 'TotalTime', 'DatePublished', 'Description', 'Images', 'RecipeCategory', 'Keywords', 'RecipeIngredientQuantities', 'RecipeIngredientParts', 'AggregatedRating', 'ReviewCount', 'Calories', 'FatContent', 'SaturatedFatContent', 'CholesterolContent', 'SodiumContent', 'CarbohydrateContent', 'FiberContent', 'SugarContent', 'ProteinContent', 'RecipeServings', 'RecipeYield', 'RecipeInstructions']


In [24]:
diff = df["n_parts"] - df["n_quants"]
print(diff.value_counts().head(10))
print("quants shorter:", (diff > 0).mean(), "| quants longer:", (diff < 0).mean())
df[df["mismatch"]].sample(5, random_state=1)[["RecipeIngredientParts", "RecipeIngredientQuantities"]]

0    84905
Name: count, dtype: int64
quants shorter: 0.0 | quants longer: 0.0


ValueError: a must be greater than 0 unless no samples are taken

In [25]:
tmp = recipes[["RecipeIngredientParts", "RecipeIngredientQuantities"]].dropna().copy()
tmp["n_parts"] = tmp["RecipeIngredientParts"].apply(parse_r_vector).str.len()
tmp["n_quants"] = tmp["RecipeIngredientQuantities"].apply(parse_r_vector).str.len()
tmp["diff"] = tmp["n_parts"] - tmp["n_quants"]

print(tmp["diff"].value_counts().head(10))
print("quants shorter:", round((tmp["diff"] > 0).mean(), 4),
      "| quants longer:", round((tmp["diff"] < 0).mean(), 4))

tmp[tmp["diff"] != 0].sample(5, random_state=1)[["RecipeIngredientParts", "RecipeIngredientQuantities"]]

diff
-1    159306
-2    121697
 0    114978
-3     64055
-4     26483
 1     18017
-5      9429
-6      3118
 2      3040
-7      1045
Name: count, dtype: int64
quants shorter: 0.0417 | quants longer: 0.7383


,RecipeIngredientParts,RecipeIngredientQuantities
287437,"c(""whole wheat flour"", ""oats"", ""golden raisin"", ""carrot"", ""flax seeds"", ""pec...","c(""1"", ""4"", ""1/2"", ""1"", ""1/4"", ""3"", ""1"", ""1/2"", ""1/8"", ""3/4"", ""2"")"
443993,"c(""pomegranate juice"", ""lime juice"", ""water"")","c(""1/2"", ""1"", ""1 1/2"", NA)"
396793,"c(""onion"", ""all-purpose flour"", ""hot mustard"", ""wine vinegar"", ""caraway seed...","c(""2"", ""1"", ""3"", ""1 1/4"", ""1 1/2"", ""5"", ""2"", ""1"", ""1"", ""1"", ""2"", ""1"", ""8"", ""..."
284397,"c(""eggs"", ""cinnamon"", ""nutmeg"", ""confectioners' sugar"")","c(""3/4"", ""3/4"", ""1"", ""4"", ""1/2"", ""1/2"", ""1"", ""1/4"")"
209548,"c(""cream cheese"", ""tuna"", ""pimientos"", ""green onions"", ""gelatin"", ""warm wate...","c(""2 -8"", ""2"", ""1"", ""1 1/2"", ""2"", ""1"", ""1"", ""1/2"", ""1"", ""1/2"", ""1/2"")"


In [26]:
import os
print(os.getcwd())
print(os.path.abspath("recipes_clean.pkl"))
print(os.path.exists("recipes_clean.pkl"))

/Users/sarah/Desktop/Final Project/my-project/notebooks
/Users/sarah/Desktop/Final Project/my-project/notebooks/recipes_clean.pkl
True


In [27]:
for kw in ["almond", "orange juice", "pumpkin seed", "chia"]:
    in_parts = recipes["RecipeIngredientParts"].str.contains(kw, case=False, na=False)
    in_name = recipes["Name"].str.contains(kw, case=False, na=False)
    print(f"{kw:14} in parts: {in_parts.sum():6} | in name: {in_name.sum():6} | in name but NOT in parts: {(in_name & ~in_parts).sum()}")

almond         in parts:    840 | in name:   4153 | in name but NOT in parts: 3787
orange juice   in parts:      1 | in name:     70 | in name but NOT in parts: 70
pumpkin seed   in parts:      1 | in name:    173 | in name but NOT in parts: 173
chia           in parts:      2 | in name:    201 | in name but NOT in parts: 201
